In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!apt-get install -y -qq libopenmpi-dev openmpi-bin > /dev/null
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-c934cb17-0e64-7864-4202-9243f0f8a985, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [2]:
# Colab prep. IMPORTANT: never import mpi4py in the notebook kernel (it breaks mpirun).
import os, subprocess
for k in list(os.environ):  # remove any leaked MPI state
    if k.startswith(("PMIX_", "OMPI_MCA_ess", "OMPI_MCA_orte", "OMPI_MCA_pmix", "OMPI_APP", "OMPI_COMM", "OMPI_FIRST", "OMPI_NUM", "OMPI_UNIVERSE")):
        del os.environ[k]
os.environ["OMPI_MCA_btl_vader_single_copy_mechanism"] = "none"
os.environ["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
os.environ["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"
r = subprocess.run(["python3", "-c", "from numba import cuda; assert cuda.is_available(); print('GPU:', cuda.get_current_device().name)"], capture_output=True, text=True)
print(r.stdout, r.stderr)
r = subprocess.run(["mpirun","--allow-run-as-root","--oversubscribe","-np","4","hostname"], capture_output=True, text=True)
print("mpirun test exit code:", r.returncode); print(r.stdout, r.stderr)

GPU: Tesla T4
 
mpirun test exit code: 0
df94513258cd
df94513258cd
df94513258cd
df94513258cd
 


In [4]:
# Replace with your exact GitHub username before running the solver
GITHUB_USERNAME = "ninielilililili"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "You must set GITHUB_USERNAME to your real GitHub handle."
)
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: ninielilililili


In [5]:
%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import re
import time
import hashlib
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

# Compile-time constants for 2D shared-memory tiling (16x16 interior + 1-cell halo apron)
TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15

# Local geometry (2x2 process grid)
NY_LOC = GLOBAL_NY // 2   # 256
NX_LOC = GLOBAL_NX // 2   # 256
PAD = NY_LOC + 2          # 258
NBUF = 2 * NX_LOC + 2 * NY_LOC  # packed halo vector length (1024)
OFF_UP, OFF_DOWN = 0, NX_LOC
OFF_LEFT, OFF_RIGHT = 2 * NX_LOC, 2 * NX_LOC + NY_LOC

CFL_FRACTION = 0.8
REPORT_EVERY = 100
TAG_UP, TAG_DOWN, TAG_LEFT, TAG_RIGHT = 0, 1, 2, 3

DX = 1.0 / (GLOBAL_NX + 1)   # 1/513
DY = 1.0 / (GLOBAL_NY + 1)


def get_gpu_uuid() -> str:
    """Query active NVIDIA GPU UUID via nvidia-smi."""
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    """Derive deterministic 64-bit integer seed bound to student GitHub handle and GPU UUID."""
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf8")).hexdigest()
    return int(digest[:16], 16)


# ==============================================================================
# TASK 1: Distributed Cartesian Topology & Hardware Context Initialization
# ==============================================================================
def init_distributed_context(github_user: str):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks (2x2 grid), got {size}.")

    # Bind process to GPU 0 on Google Colab
    cuda.select_device(0)
    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)

    cart = comm.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    crank = cart.Get_rank()
    row_rank, col_rank = cart.Get_coords(crank)
    nbr_up, nbr_down = cart.Shift(0, 1)     # (source, dest) along axis 0
    nbr_left, nbr_right = cart.Shift(1, 1)  # (source, dest) along axis 1

    return {
        "world": comm, "cart": cart, "rank": crank,
        "row_rank": row_rank, "col_rank": col_rank,
        "nbr_up": nbr_up, "nbr_down": nbr_down,
        "nbr_left": nbr_left, "nbr_right": nbr_right,
        "gpu_uuid": gpu_uuid, "seed": instance_seed, "github_user": github_user,
    }


# ==============================================================================
# TASK 2: Seeded heterogeneous subdomain initialization (rank-local only)
# ==============================================================================
def initialize_subdomains(ctx):
    """Return host arrays (u0, kappa, S), each (258, 258), incl. ghost indices 0 and 257."""
    rr, cr = ctx["row_rank"], ctx["col_rank"]
    # Global node index of local index i is rr*ny + i  (i=1 on rank 0 -> node 1; 0 and 513 are walls)
    y = (rr * NY_LOC + np.arange(PAD)) * DY
    x = (cr * NX_LOC + np.arange(PAD)) * DX
    Y, X = np.meshgrid(y, x, indexing="ij")

    rng = np.random.default_rng(ctx["seed"])  # same stream on every rank -> consistent fields
    phi1, phi2 = rng.uniform(0.0, 2.0 * math.pi, size=2)
    n_pulses = 3
    centers = rng.uniform(0.25, 0.75, size=(n_pulses, 2))
    amps = rng.uniform(0.8, 1.5, size=n_pulses)
    sigma = 0.07
    src_c = rng.uniform(0.3, 0.7, size=2)

    # kappa in [0.08, 0.24]
    kappa = 0.16 + 0.08 * np.sin(3.0 * math.pi * X + phi1) * np.cos(2.0 * math.pi * Y + phi2)

    u = np.zeros((PAD, PAD), dtype=np.float64)
    for (cx, cy), a in zip(centers, amps):
        u += a * np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2.0 * sigma ** 2))

    # Small localized source (kept weak so damping dominates energy balance)
    S = 0.002 * np.exp(-((X - src_c[0]) ** 2 + (Y - src_c[1]) ** 2) / (2.0 * 0.05 ** 2))

    # Dirichlet u = 0 on global outer edges (ghost layer where neighbor is PROC_NULL)
    if ctx["nbr_up"] == MPI.PROC_NULL:
        u[0, :] = 0.0
    if ctx["nbr_down"] == MPI.PROC_NULL:
        u[PAD - 1, :] = 0.0
    if ctx["nbr_left"] == MPI.PROC_NULL:
        u[:, 0] = 0.0
    if ctx["nbr_right"] == MPI.PROC_NULL:
        u[:, PAD - 1] = 0.0

    return (np.ascontiguousarray(u), np.ascontiguousarray(kappa), np.ascontiguousarray(S))


# ==============================================================================
# TASK 3: GPU pack/unpack kernels & pinned host-staged exchange
# ==============================================================================
@cuda.jit
def pack_halos_kernel(u, sbuf, ny, nx):
    """sbuf layout: [row 1 (nx) | row ny (nx) | col 1 (ny) | col nx (ny)]."""
    t = cuda.grid(1)
    if t < nx:
        sbuf[t] = u[1, t + 1]
    elif t < 2 * nx:
        k = t - nx
        sbuf[t] = u[ny, k + 1]
    elif t < 2 * nx + ny:
        k = t - 2 * nx
        sbuf[t] = u[k + 1, 1]
    elif t < 2 * nx + 2 * ny:
        k = t - 2 * nx - ny
        sbuf[t] = u[k + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, rbuf, ny, nx, mask):
    """rbuf layout: [from up -> ghost row 0 | from down -> row ny+1 | from left -> col 0 | from right -> col nx+1].
    mask bits: 1 up, 2 down, 4 left, 8 right (neighbor exists)."""
    t = cuda.grid(1)
    if t < nx:
        if mask & 1:
            u[0, t + 1] = rbuf[t]
    elif t < 2 * nx:
        k = t - nx
        if mask & 2:
            u[ny + 1, k + 1] = rbuf[t]
    elif t < 2 * nx + ny:
        k = t - 2 * nx
        if mask & 4:
            u[k + 1, 0] = rbuf[t]
    elif t < 2 * nx + 2 * ny:
        k = t - 2 * nx - ny
        if mask & 8:
            u[k + 1, nx + 1] = rbuf[t]


class RankState:
    """All rank-local device/host resources (no global arrays)."""

    def __init__(self, ctx, u0, kappa, S):
        self.d_u = [cuda.to_device(u0), cuda.to_device(u0.copy())]
        self.cur = 0
        self.d_kappa = cuda.to_device(kappa)
        self.d_S = cuda.to_device(S)
        self.d_send = cuda.device_array(NBUF, dtype=np.float64)
        self.d_recv = cuda.device_array(NBUF, dtype=np.float64)
        self.h_send = cuda.pinned_array(NBUF, dtype=np.float64)
        self.h_recv = cuda.pinned_array(NBUF, dtype=np.float64)
        self.h_send[:] = 0.0
        self.h_recv[:] = 0.0
        self.stream_boundary = cuda.stream()
        self.stream_interior = cuda.stream()
        self.mask = (
            (1 if ctx["nbr_up"] != MPI.PROC_NULL else 0)
            | (2 if ctx["nbr_down"] != MPI.PROC_NULL else 0)
            | (4 if ctx["nbr_left"] != MPI.PROC_NULL else 0)
            | (8 if ctx["nbr_right"] != MPI.PROC_NULL else 0)
        )


PACK_BLOCK = 256
PACK_GRID = (NBUF + PACK_BLOCK - 1) // PACK_BLOCK


def begin_exchange(ctx, st, stream):
    """Stage A + posting of Stage B: pack on GPU, async D2H, sync, post Irecv/Isend."""
    d_u = st.d_u[st.cur]
    pack_halos_kernel[PACK_GRID, PACK_BLOCK, stream](d_u, st.d_send, NY_LOC, NX_LOC)
    st.d_send.copy_to_host(st.h_send, stream=stream)
    stream.synchronize()

    cart = ctx["cart"]
    hs, hr = st.h_send, st.h_recv
    reqs = []
    # Receives: tag = direction of travel of the message
    reqs.append(cart.Irecv(hr[OFF_UP:OFF_UP + NX_LOC], source=ctx["nbr_up"], tag=TAG_DOWN))
    reqs.append(cart.Irecv(hr[OFF_DOWN:OFF_DOWN + NX_LOC], source=ctx["nbr_down"], tag=TAG_UP))
    reqs.append(cart.Irecv(hr[OFF_LEFT:OFF_LEFT + NY_LOC], source=ctx["nbr_left"], tag=TAG_RIGHT))
    reqs.append(cart.Irecv(hr[OFF_RIGHT:OFF_RIGHT + NY_LOC], source=ctx["nbr_right"], tag=TAG_LEFT))
    # Sends
    reqs.append(cart.Isend(hs[OFF_UP:OFF_UP + NX_LOC], dest=ctx["nbr_up"], tag=TAG_UP))
    reqs.append(cart.Isend(hs[OFF_DOWN:OFF_DOWN + NX_LOC], dest=ctx["nbr_down"], tag=TAG_DOWN))
    reqs.append(cart.Isend(hs[OFF_LEFT:OFF_LEFT + NY_LOC], dest=ctx["nbr_left"], tag=TAG_LEFT))
    reqs.append(cart.Isend(hs[OFF_RIGHT:OFF_RIGHT + NY_LOC], dest=ctx["nbr_right"], tag=TAG_RIGHT))
    return reqs


def finish_exchange(ctx, st, reqs, stream):
    """Stage C (part 1): wait, async H2D, unpack into ghost rows/cols."""
    MPI.Request.Waitall(reqs)
    st.d_recv.copy_to_device(st.h_recv, stream=stream)
    unpack_halos_kernel[PACK_GRID, PACK_BLOCK, stream](
        st.d_u[st.cur], st.d_recv, NY_LOC, NX_LOC, st.mask
    )


def exchange_halos_async(ctx, st, stream):
    """Full halo exchange on one stream (used by the serialized path)."""
    reqs = begin_exchange(ctx, st, stream)
    finish_exchange(ctx, st, reqs, stream)


# ==============================================================================
# TASK 4: Shared-memory tiled stencil & dual-stream orchestrator
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u_in, u_out, kappa, src, r_start, r_end, c_start, c_end,
                         dt, inv_dx2, inv_dy2, lam):
    """Updates cells i in [r_start, r_end], j in [c_start, c_end] (inclusive).
    Launched with 16x16 blocks; every thread reaches syncthreads (no early return)."""
    smem = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = r_start + cuda.blockIdx.y * TILE_DIM + ty
    j = c_start + cuda.blockIdx.x * TILE_DIM + tx
    H = u_in.shape[0]
    W = u_in.shape[1]
    in_arr = (i < H) and (j < W)

    if in_arr:
        smem[ty + 1, tx + 1] = u_in[i, j]
        if ty == 0:
            smem[0, tx + 1] = u_in[i - 1, j]
        if ty == TILE_DIM - 1 and i + 1 < H:
            smem[SHARED_DIM - 1, tx + 1] = u_in[i + 1, j]
        if tx == 0:
            smem[ty + 1, 0] = u_in[i, j - 1]
        if tx == TILE_DIM - 1 and j + 1 < W:
            smem[ty + 1, SHARED_DIM - 1] = u_in[i, j + 1]

    cuda.syncthreads()

    if i >= r_start and i <= r_end and j >= c_start and j <= c_end:
        c = smem[ty + 1, tx + 1]
        n = smem[ty, tx + 1]
        s = smem[ty + 2, tx + 1]
        w = smem[ty + 1, tx]
        e = smem[ty + 1, tx + 2]
        k0 = kappa[i, j]
        kn = 0.5 * (kappa[i - 1, j] + k0)
        ks = 0.5 * (kappa[i + 1, j] + k0)
        kw = 0.5 * (kappa[i, j - 1] + k0)
        ke = 0.5 * (kappa[i, j + 1] + k0)
        du = ((ks * (s - c) - kn * (c - n)) * inv_dy2
              + (ke * (e - c) - kw * (c - w)) * inv_dx2
              - lam * c * c * c + src[i, j])
        u_out[i, j] = c + dt * du


def launch_box(st, stream, r0, r1, c0, c1, dt):
    nby = (r1 - r0 + 1 + TILE_DIM - 1) // TILE_DIM
    nbx = (c1 - c0 + 1 + TILE_DIM - 1) // TILE_DIM
    tiled_stencil_kernel[(nbx, nby), (TILE_DIM, TILE_DIM), stream](
        st.d_u[st.cur], st.d_u[1 - st.cur], st.d_kappa, st.d_S,
        r0, r1, c0, c1, dt, 1.0 / (DX * DX), 1.0 / (DY * DY), LAMBDA_DAMP,
    )


def step_serialized(ctx, st, dt):
    sb = st.stream_boundary
    exchange_halos_async(ctx, st, sb)
    launch_box(st, sb, 1, NY_LOC, 1, NX_LOC, dt)
    sb.synchronize()
    st.cur = 1 - st.cur


def step_overlapped(ctx, st, dt):
    sb, si = st.stream_boundary, st.stream_interior
    # Stage A + B: pack, D2H, sync boundary stream, post non-blocking MPI
    reqs = begin_exchange(ctx, st, sb)
    # Deep interior concurrently with communication
    launch_box(st, si, 2, NY_LOC - 1, 2, NX_LOC - 1, dt)
    # Stage C: wait, H2D, unpack, boundary ring
    finish_exchange(ctx, st, reqs, sb)
    launch_box(st, sb, 1, 1, 1, NX_LOC, dt)                     # row 1
    launch_box(st, sb, NY_LOC, NY_LOC, 1, NX_LOC, dt)           # row ny
    launch_box(st, sb, 2, NY_LOC - 1, 1, 1, dt)                 # col 1
    launch_box(st, sb, 2, NY_LOC - 1, NX_LOC, NX_LOC, dt)       # col nx
    si.synchronize()
    sb.synchronize()
    st.cur = 1 - st.cur


# ==============================================================================
# TASK 5: Reductions, PTX audit, benchmark
# ==============================================================================
def global_diagnostics(ctx, st):
    cart = ctx["cart"]
    new = st.d_u[st.cur].copy_to_host()[1:NY_LOC + 1, 1:NX_LOC + 1]
    old = st.d_u[1 - st.cur].copy_to_host()[1:NY_LOC + 1, 1:NX_LOC + 1]
    loc = np.array([new.sum() * DX * DY, ((new - old) ** 2).sum() * DX * DY])
    tot = np.zeros(2)
    cart.Allreduce(loc, tot, op=MPI.SUM)
    return float(tot[0]), float(math.sqrt(tot[1]))


def run_mode(ctx, host, mode, dt, steps, diagnostics=True):
    u0, kappa, S = host
    st = RankState(ctx, u0, kappa, S)
    step_fn = step_overlapped if mode == "overlapped" else step_serialized
    start_ev, end_ev = cuda.event(), cuda.event()

    energy = {}
    if diagnostics:
        loc = np.array([u0[1:-1, 1:-1].sum() * DX * DY])
        tot = np.zeros(1)
        ctx["cart"].Allreduce(loc, tot, op=MPI.SUM)
        energy[0] = {"energy": float(tot[0]), "residual_l2": 0.0}

    ctx["cart"].Barrier()
    cuda.synchronize()
    wall = 0.0
    event_ms = 0.0
    for n in range(1, steps + 1):
        t0 = time.perf_counter()
        start_ev.record(st.stream_boundary)
        step_fn(ctx, st, dt)
        end_ev.record(st.stream_boundary)
        end_ev.synchronize()
        wall += time.perf_counter() - t0
        event_ms += cuda.event_elapsed_time(start_ev, end_ev)
        if diagnostics and n % REPORT_EVERY == 0:
            e, r = global_diagnostics(ctx, st)
            energy[n] = {"energy": e, "residual_l2": r}

    final = st.d_u[st.cur].copy_to_host()[1:NY_LOC + 1, 1:NX_LOC + 1].copy()
    return {"final": final, "energy": energy, "wall_ms": wall * 1e3, "event_ms": event_ms}


def ptx_audit():
    asm = tiled_stencil_kernel.inspect_asm()
    ptx = "\n".join(asm.values()) if isinstance(asm, dict) else str(asm)
    counts = {
        "st.shared.f64": len(re.findall(r"\bst\.shared\.f64\b", ptx)),
        "ld.shared.f64": len(re.findall(r"\bld\.shared\.f64\b", ptx)),
        "st.shared_any": len(re.findall(r"\bst\.shared\b", ptx)),
        "ld.shared_any": len(re.findall(r"\bld\.shared\b", ptx)),
        "bar.sync": len(re.findall(r"\bbar\.sync\b", ptx)),
    }
    passed = counts["st.shared_any"] > 0 and counts["ld.shared_any"] > 0 and counts["bar.sync"] > 0
    return counts, passed


def rank_max(ctx, value):
    out = np.zeros(1)
    ctx["cart"].Allreduce(np.array([float(value)]), out, op=MPI.MAX)
    return float(out[0])


def run_solver(ctx):
    rank = ctx["rank"]
    host = initialize_subdomains(ctx)

    kappa_max = rank_max(ctx, host[1][1:-1, 1:-1].max())
    dt_max = DX * DX / (4.0 * kappa_max)
    dt = CFL_FRACTION * dt_max

    # JIT warm-up so compile time does not pollute timings (also needed before inspect_asm)
    for mode in ("serialized", "overlapped"):
        run_mode(ctx, host, mode, dt, 3, diagnostics=False)

    res_s = run_mode(ctx, host, "serialized", dt, STEPS)
    res_o = run_mode(ctx, host, "overlapped", dt, STEPS)

    parity = rank_max(ctx, np.abs(res_s["final"] - res_o["final"]).max())
    serialized_ms = rank_max(ctx, res_s["wall_ms"])
    overlapped_ms = rank_max(ctx, res_o["wall_ms"])
    ser_ev = rank_max(ctx, res_s["event_ms"])
    ovl_ev = rank_max(ctx, res_o["event_ms"])

    counts, passed = ptx_audit()

    if rank == 0:
        print(f"MPI ranks: {ctx['world'].Get_size()} in 2x2 Cartesian grid (non-periodic)")
        print(f"GitHub user: {ctx['github_user']}")
        print(f"GPU UUID: {ctx['gpu_uuid']}")
        print(f"INSTANCE_SEED: {ctx['seed']}")
        print(f"kappa_max = {kappa_max:.6f}, dx = 1/513 = {DX:.8e}")
        print(f"dt_max = dx^2/(4 kappa_max) = {dt_max:.6e}; dt used = {dt:.6e} ({CFL_FRACTION} * dt_max)")
        print("Allreduce energy / L2 residual trace (overlapped run):")
        for n in sorted(res_o["energy"]):
            d = res_o["energy"][n]
            print(f"  step {n:4d}: E = {d['energy']:.12e}   ||du||_2 = {d['residual_l2']:.6e}")
        print(f"serialized_ms = {serialized_ms:.3f} (CUDA events: {ser_ev:.3f})")
        print(f"overlapped_ms = {overlapped_ms:.3f} (CUDA events: {ovl_ev:.3f})")
        print(f"overlap speedup = {serialized_ms / overlapped_ms:.4f}x")
        print(f"parity_max_diff = {parity:.3e}")
        print(f"PTX counts: {counts}  -> ptx_audit_passed = {passed}")

        manifest = {
            "github_username": ctx["github_user"],
            "gpu_uuid": ctx["gpu_uuid"],
            "instance_seed": ctx["seed"],
            "mpi_ranks": ctx["world"].Get_size(),
            "cartesian_dims": [2, 2],
            "global_grid": [GLOBAL_NY, GLOBAL_NX],
            "local_padded_shape": [PAD, PAD],
            "steps": STEPS,
            "kappa_max": kappa_max,
            "dx": DX,
            "dt": dt,
            "dt_max": dt_max,
            "energy_trace": {str(k): v for k, v in res_o["energy"].items()},
            "energy_trace_serialized": {str(k): v for k, v in res_s["energy"].items()},
            "serialized_ms": serialized_ms,
            "overlapped_ms": overlapped_ms,
            "serialized_cuda_event_ms": ser_ev,
            "overlapped_cuda_event_ms": ovl_ev,
            "ptx_counts": counts,
            "ptx_audit_passed": passed,
            "parity_max_diff": parity,
        }
        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        print("Wrote submission_manifest.json")
        if parity >= 1e-12 or not passed:
            print("WARNING: verification criteria not met (parity < 1e-12 and PTX audit required).")


def main():
    if len(sys.argv) < 2:
        raise SystemExit("usage: distributed_cuda_solver.py <github_username>")
    ctx = init_distributed_context(sys.argv[1])
    run_solver(ctx)


if __name__ == "__main__":
    main()


Writing distributed_cuda_solver.py


In [6]:
import subprocess
cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")

MPI ranks: 4 in 2x2 Cartesian grid (non-periodic)
GitHub user: ninielilililili
GPU UUID: GPU-c934cb17-0e64-7864-4202-9243f0f8a985
INSTANCE_SEED: 14497803519286042718
kappa_max = 0.240000, dx = 1/513 = 1.94931774e-03
dt_max = dx^2/(4 kappa_max) = 3.958166e-06; dt used = 3.166533e-06 (0.8 * dt_max)
Allreduce energy / L2 residual trace (overlapped run):
  step    0: E = 9.304338029534e-02   ||du||_2 = 0.000000e+00
  step  100: E = 9.304075972439e-02   ||du||_2 = 3.493217e-05
  step  200: E = 9.303845265112e-02   ||du||_2 = 3.368144e-05
  step  300: E = 9.303619041958e-02   ||du||_2 = 3.250615e-05
  step  400: E = 9.303393292585e-02   ||du||_2 = 3.139945e-05
serialized_ms = 1117.875 (CUDA events: 1040.535)
overlapped_ms = 1525.892 (CUDA events: 1457.040)
overlap speedup = 0.7326x
parity_max_diff = 0.000e+00
PTX counts: {'st.shared.f64': 5, 'ld.shared.f64': 5, 'st.shared_any': 5, 'ld.shared_any': 5, 'bar.sync': 1}  -> ptx_audit_passed = True
Wrote submission_manifest.json



Q1. Memory Coalescing

The array is stored in row-major order, so column elements are not contiguous. Direct column copying transfers unnecessary data. pack_halos_kernel collects only the required elements into a continuous buffer. Each rank transfers 8 KB of halo data per step.

Q2. Shared Memory

Threads load the centre and four neighbouring elements. Row-wise access is coalesced and avoids bank conflicts. Column access can cause potential two-way conflicts, but the actual apron access pattern avoids problematic conflicts.

Q3. PTX Audit

The PTX audit passed successfully:
- Shared stores: 5
- Shared loads: 5
- Barriers: 1
The barrier (bar.sync) synchronizes all threads before stencil calculations, preventing threads from reading unfinished shared-memory data.

Q4. GPU Context Sharing

Four MPI processes share one Tesla T4 GPU. Overlapping execution was slower:
- Serialized: 1117.875 ms
- Overlapped: 1525.892 ms
- Speedup: 0.7326×
Extra scheduling and context-switching overhead reduced performance.

Q5. MPI Deadlock

Blocking sends can cause deadlocks when processes wait for each other's receives. Non-blocking MPI.Isend() and MPI.Irecv() with MPI.Waitall() avoid this problem. MPI.PROC_NULL handles missing neighbours without additional communication branches.

Q6. Stability and Energy

The maximum stable time step is 3.958166e-06 s. The simulation uses 3.166533e-06 s, satisfying the CFL stability condition.

Energy decreases from 0.09304338 to 0.09303393 over 400 steps. The decreasing L2 residual also indicates stable behaviour.
